<a href="https://colab.research.google.com/github/dieynabadiop88-del/Dieynaba_INFO4670_Fall2026/blob/main/Copy_of_Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [3]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [4]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = None
missing = students.isna().sum()
print(missing[missing > 0])
n_missing_study = students["study_hours_reported"].isna().sum()
print(f"\nstudy_hours_reported: {n_missing_study} blank of {len(students)} "
      f"({n_missing_study/len(students)*100:.1f}%) -> {students['study_hours_reported'].notna().sum()} present")

study_hours_reported    255
dtype: int64

study_hours_reported: 255 blank of 2027 (12.6%) -> 1772 present


**Your justification (1–2 sentences):** _I would use the mean or median because it keeps the dataset from losing rows and provides a reasonable estimate for the missing study_hours_reported values. The median is especially appropriate if the data is skewed because it is less affected by unusually high or low study hours._

In [5]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [7]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
print("RAW housing — eight spellings for three groups:")
print(students["housing"].value_counts())

clean_map = {"on-campus":"On-Campus", "off-campus":"Off-Campus",
             "off campus":"Off-Campus", "with family":"With Family"}
students["housing_clean"] = students["housing"].str.strip().str.lower().map(clean_map)

print("\nCLEANED — three real groups:")
print(students["housing_clean"].value_counts())

RAW housing — eight spellings for three groups:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

CLEANED — three real groups:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [8]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [9]:
# TODO
impossible_ages = None
n_neg_work = None
impossible_ages = sorted(int(a) for a in students.loc[(students["age"] < 15) | (students["age"] > 90), "age"].unique())
n_neg_work = (students["work_hours_per_week"] < 0).sum()
print("impossible ages:", impossible_ages)
print("students with negative work hours:", n_neg_work)

# The 43 zero-GPAs are ambiguous, not impossible.
zeros = students["final_gpa"] == 0
print("\nGPA = 0.00 rows:", int(zeros.sum()),
      "| of those, marked dropped_out:", int(students.loc[zeros, "dropped_out"].sum()))

impossible ages: [-22, 0, 1, 3, 199, 220]
students with negative work hours: 4

GPA = 0.00 rows: 43 | of those, marked dropped_out: 9


In [10]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [30]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset=["student_id"])
# student_records SHOULD be one row per student:
print("student_records rows:", len(students), "| unique student_id:", students["student_id"].nunique())
print("exact duplicate rows:", students.duplicated().sum(),
      "| duplicate student_ids:", students["student_id"].duplicated().sum())

# enroll / activity have MANY rows per student BY DESIGN -> not duplicates:
print("\nenroll : one row per enrollment ->", len(enroll), "rows for", enroll["sid"].nunique(), "students")
print("activity: one row per student-week ->", len(activity), "rows for",
      activity["student_id"].nunique(), "students x", activity["week"].nunique(), "weeks")

student_records rows: 2027 | unique student_id: 2000
exact duplicate rows: 18 | duplicate student_ids: 27

enroll : one row per enrollment -> 8088 rows for 2014 students
activity: one row per student-week -> 32000 rows for 2000 students x 16 weeks


**Why not de-dupe enroll / activity? (1 sentence):** _We must not de-duplicate enroll or activity by ID because multiple rows for the same student represent valid enrollments or weekly activity records._

In [31]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [49]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
print("student_records key:", students_dedup["student_id"].iloc[0], "->", students_dedup["student_id"].dtype)
print("course_enroll  key:", enroll["sid"].iloc[0], "->", enroll["sid"].dtype)

key = students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int)

activity_summary = activity.groupby("student_id")["minutes_active"].sum().reset_index()

analysis = pd.merge(
    students_dedup.assign(student_key=key),
    activity_summary,
    on="student_id",
    how="left"
)
# (1) As-is: text vs number -> pandas refuses.
try:
    pd.merge(students, enroll, left_on="student_id", right_on="sid")
except Exception as e:
    print("\nJoin as-is FAILS:", type(e).__name__, "-", str(e)[:55])

# (2) Both as text: it runs, but nothing matches ("NU-101508" != "108933").
both = pd.merge(students.assign(k=students["student_id"].astype(str)),
                enroll.assign(k=enroll["sid"].astype(str)), on="k")
print("Both as text -> rows matched:", len(both))

student_records key: NU-101508 -> object
course_enroll  key: 108933 -> int64

Join as-is FAILS: ValueError - You are trying to merge on object and int64 columns for
Both as text -> rows matched: 0


In [50]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [51]:
# TODO
matched = None
# (3) Standardize: strip "NU-", match on the number.
key = students["student_id"].str.replace("NU-", "", regex=False).astype(int)

matched      = enroll["sid"].isin(set(key)).sum()
orphan_sids  = set(enroll["sid"]) - set(key)
print("enrollment rows that match a student:", matched, "of", len(enroll))
print("orphan IDs (no student record):", len(orphan_sids),
      "-> orphan rows:", enroll["sid"].isin(orphan_sids).sum())

# The deferred check: do the 43 zero-GPA students actually exist elsewhere?
zero_keys = set(key[students["final_gpa"].values == 0])
act_keys  = set(activity["student_id"].str.replace("NU-", "", regex=False).astype(int))
print("\n43 zeros with an enrollment:", len(zero_keys & set(enroll["sid"])),
      "| with activity:", len(zero_keys & act_keys))

enrollment rows that match a student: 8041 of 8088
orphan IDs (no student record): 14 -> orphan rows: 47

43 zeros with an enrollment: 43 | with activity: 43


In [52]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [53]:
# TODO
dates_parsed = pd.to_datetime(
    students["enrollment_date"],
    format="mixed",
    errors="coerce"
)

import re
def date_format(s):
    if re.match(r"^\d{4}-\d{2}-\d{2}$", s):        return "YYYY-MM-DD"
    if re.match(r"^\d{1,2}/\d{1,2}/\d{4}$", s):     return "M/D/YYYY"
    if re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}$", s):  return "D-Mon-YYYY"
    return "other"
print(students["enrollment_date"].map(date_format).value_counts())

naive      = pd.to_datetime(students["enrollment_date"], errors="coerce")             # the trap
deliberate = pd.to_datetime(students["enrollment_date"], format="mixed", errors="coerce")
print("\nnaive parse      -> blanks (NaT):", naive.isna().sum())
print("deliberate parse -> blanks (NaT):", deliberate.isna().sum())

enrollment_date
M/D/YYYY      1165
YYYY-MM-DD     557
D-Mon-YYYY     305
Name: count, dtype: int64

naive parse      -> blanks (NaT): 1470
deliberate parse -> blanks (NaT): 0


In [54]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [57]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]


# On our data: z-score the columns a distance-based method would use.
for col in ["commute_miles", "study_hours_reported"]:
    z = (students[col] - students[col].mean()) / students[col].std()
    print(f"{col}: after z-score  mean = {z.mean():.2f},  sd = {z.std():.2f}")

analysis["study_z"] = (
    analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()
) / analysis["study_hours_reported"].std()

#Discretization
bands = pd.cut(students["final_gpa"], bins=[-0.01, 1, 2, 3, 4],
               labels=["0-1", "1-2", "2-3", "3-4"])

analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=["0-1", "1-2", "2-3", "3-4"]
)


print(bands.value_counts().sort_index())

commute_miles: after z-score  mean = -0.00,  sd = 1.00
study_hours_reported: after z-score  mean = 0.00,  sd = 1.00
final_gpa
0-1      71
1-2     532
2-3    1118
3-4     306
Name: count, dtype: int64


In [58]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [59]:
# TODO: write analysis to northgate_clean.csv
clean = students.copy()                                   # never touch the raw

# 1. de-duplicate (student_records only: one row per student)
clean = clean.drop_duplicates()                          # 18 exact copies
clean = clean.drop_duplicates(subset="student_id")       # 9 near-duplicates -> keep first

# 2. standardize categories (already computed as housing_clean)
# 3. fix impossible values -> mark missing
clean.loc[(clean["age"] < 15) | (clean["age"] > 90), "age"] = np.nan
clean.loc[clean["work_hours_per_week"] < 0, "work_hours_per_week"] = np.nan
# 4. parse dates deliberately
clean["enrollment_date"] = pd.to_datetime(clean["enrollment_date"], format="mixed", errors="coerce")
# 5. standardize the key for joining
clean["sid"] = clean["student_id"].str.replace("NU-", "", regex=False).astype(int)

print("clean student table:", clean.shape)
analysis.to_csv("northgate_clean.csv", index=False)   # a NEW file; raw stays untouched
print("wrote northgate_clean.csv")

clean student table: (2000, 14)
wrote northgate_clean.csv


In [60]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing values → Kept valid values and treated invalid/unavailable values as
missing rather than inventing replacements.
- Housing → Standardized housing categories so different labels for the same housing type are grouped consistently.
- Impossible values → Replaced impossible ages and negative work hours with NaN because they are not valid observations.
- Duplicates → Removed exact duplicates and then removed near-duplicate student records using student_id, keeping one row per student.
- Key → Removed the "NU-" prefix and converted the student ID to an integer so it could be matched consistently with enrollment/activity data.
- Dates → Parsed enrollment_date using format="mixed" so valid dates in different formats were preserved.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [66]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
# Mean GPA
mean_gpa = analysis["final_gpa"].mean()
print("Mean GPA:", mean_gpa)

# One relationship: study hours and GPA
relationship = analysis[["study_hours_reported", "final_gpa"]].corr()
print("\nRelationship between study hours and GPA:")
print(relationship)

# Compare with raw data
raw_mean_gpa = students["final_gpa"].mean()
print("\nRaw mean GPA:", raw_mean_gpa)
print("Clean mean GPA:", mean_gpa)

Mean GPA: 2.3053600000000003

Relationship between study hours and GPA:
                      study_hours_reported  final_gpa
study_hours_reported              1.000000   0.693922
final_gpa                         0.693922   1.000000

Raw mean GPA: 2.307474099654662
Clean mean GPA: 2.3053600000000003


In [64]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
